In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

df = pd.read_csv(DATA_PROCESSED / "ipo_model_ready.csv")

df.head()


,listing_date,company,issue_size_crore,qib,hni,rii,total,issue_price,listing_price,listing_gain_pct,cmpbse,cmpnse,current_gain_pct,success
0,2025-08-06,M & B Engineering Ltd,650.00,36.72,38.24,32.55,36.20,385.0,386.0,0.26,426.85,426.15,10.87,1
1,2025-08-06,Sri Lotus Developers & Realty Ltd,792.00,163.90,57.71,20.28,69.14,150.0,179.1,19.40,201.10,199.72,34.07,1
2,2025-08-06,National Securities Depository Ltd (NSDL),NaN,103.97,34.98,7.73,41.01,800.0,880.0,10.00,NaN,61.76,NaN,1
3,2025-08-05,Aditya Infotech Ltd,NaN,133.21,72.00,50.87,100.69,675.0,NaN,50.81,NaN,NaN,57.72,1
4,2025-08-05,Laxmi India Finance Ltd,254.26,1.30,1.84,2.22,1.87,158.0,136.0,-13.92,149.00,150.00,-5.70,0


In [2]:
FEATURES = [
    "issue_size_crore",
    "qib",
    "hni",
    "rii",
    "total",
    "issue_price"
]

X = df[FEATURES]
y = df["success"]


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [4]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

dt_model = DecisionTreeClassifier(
    max_depth=4,          # keeps it interpretable (important for paper)
    min_samples_leaf=10,  # avoids overfitting
    random_state=42
)

dt_model.fit(X_train, y_train)


,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",4
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current n

In [5]:
y_pred_dt = dt_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print(confusion_matrix(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))


Accuracy: 0.6902654867256637
[[13 22]
 [13 65]]
              precision    recall  f1-score   support

           0       0.50      0.37      0.43        35
           1       0.75      0.83      0.79        78

    accuracy                           0.69       113
   macro avg       0.62      0.60      0.61       113
weighted avg       0.67      0.69      0.68       113



In [6]:
import pandas as pd

feature_importance_dt = pd.DataFrame({
    "feature": FEATURES,
    "importance": dt_model.feature_importances_
}).sort_values(by="importance", ascending=False)

feature_importance_dt


,feature,importance
4,total,0.674621
0,issue_size_crore,0.127326
1,qib,0.125240
2,hni,0.040335
3,rii,0.032479
5,issue_price,0.000000


In [7]:
import joblib
from pathlib import Path

MODELS_DIR = Path.cwd() / "models"
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(dt_model, MODELS_DIR / "decision_tree_baseline.pkl")


['/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/notebooks/phase1/models/decision_tree_baseline.pkl']

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=5,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [9]:
y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


Accuracy: 0.7787610619469026
[[25 10]
 [15 63]]
              precision    recall  f1-score   support

           0       0.62      0.71      0.67        35
           1       0.86      0.81      0.83        78

    accuracy                           0.78       113
   macro avg       0.74      0.76      0.75       113
weighted avg       0.79      0.78      0.78       113



In [10]:
rf_importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

rf_importance


,feature,importance
4,total,0.321179
1,qib,0.253160
2,hni,0.170723
3,rii,0.098713
0,issue_size_crore,0.098610
5,issue_price,0.057614


In [11]:
joblib.dump(rf_model, MODELS_DIR / "random_forest_baseline.pkl")


['/Users/jeeveshmishra/Desktop/IPO-Success-Prediction/IPO-Success-Prediction/notebooks/phase1/models/random_forest_baseline.pkl']

In [12]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    rf_model,
    X,
    y,
    cv=5,
    scoring="f1"
)

cv_scores, cv_scores.mean()


(array([0.83435583, 0.84210526, 0.82608696, 0.76595745, 0.64957265]),
 np.float64(0.7836156288563306))

In [13]:
y_probs = rf_model.predict_proba(X_test)[:, 1]


In [14]:
for t in [0.3, 0.4, 0.5, 0.6]:
    y_pred_t = (y_probs >= t).astype(int)
    print(f"Threshold {t}")
    print(classification_report(y_test, y_pred_t))


Threshold 0.3
              precision    recall  f1-score   support

           0       0.61      0.40      0.48        35
           1       0.77      0.88      0.82        78

    accuracy                           0.73       113
   macro avg       0.69      0.64      0.65       113
weighted avg       0.72      0.73      0.72       113

Threshold 0.4
              precision    recall  f1-score   support

           0       0.71      0.63      0.67        35
           1       0.84      0.88      0.86        78

    accuracy                           0.81       113
   macro avg       0.78      0.76      0.76       113
weighted avg       0.80      0.81      0.80       113

Threshold 0.5
              precision    recall  f1-score   support

           0       0.62      0.71      0.67        35
           1       0.86      0.81      0.83        78

    accuracy                           0.78       113
   macro avg       0.74      0.76      0.75       113
weighted avg       0.79      0.7

In [15]:
OPTIMAL_THRESHOLD = 0.4
y_pred_final = (y_probs >= OPTIMAL_THRESHOLD).astype(int)


In [16]:
error_df = X_test.copy()
error_df["actual"] = y_test.values
error_df["predicted"] = y_pred_final
error_df["prob_success"] = y_probs
error_df["error_type"] = "Correct"

error_df.loc[
    (error_df.actual == 1) & (error_df.predicted == 0),
    "error_type"
] = "False Negative"

error_df.loc[
    (error_df.actual == 0) & (error_df.predicted == 1),
    "error_type"
] = "False Positive"

error_df["error_type"].value_counts()


error_type
Correct           91
False Positive    13
False Negative     9
Name: count, dtype: int64

In [17]:
error_df.groupby("error_type")[FEATURES].mean()


,issue_size_crore,qib,hni,rii,total,issue_price
error_type,,,,,,
Correct,1580.479231,61.158242,91.003187,17.272308,46.265934,332.023256
False Negative,1065.840000,2.325556,1.874444,1.111111,1.582222,155.444444
False Positive,1880.503077,10.804615,12.230000,3.861538,7.828462,245.333333


In [20]:
eps = 1e-6  # numerical stability

df["qib_ratio"] = df["qib"] / (df["total"] + eps)
df["hni_ratio"] = df["hni"] / (df["total"] + eps)
df["rii_ratio"] = df["rii"] / (df["total"] + eps)

df["inst_ratio"] = (df["qib"] + df["hni"]) / (df["total"] + eps)


In [21]:
df[["qib_ratio", "hni_ratio", "rii_ratio", "inst_ratio"]].describe()


,qib_ratio,hni_ratio,rii_ratio,inst_ratio
count,559.000000,559.000000,559.000000,559.000000
mean,1.400562,1.436495,0.664086,2.837057
std,0.929401,1.057051,0.698142,1.014078
min,0.000000,0.000000,0.000000,0.000000
25%,0.765144,0.589309,0.195384,2.039402
50%,1.282619,1.161290,0.401316,2.893707
75%,1.834285,2.142646,0.886986,3.489095
max,7.409522,5.138155,4.727230,7.670475


In [22]:
FEATURES_A = [
    "issue_size_crore",
    "issue_price",
    "qib_ratio",
    "hni_ratio",
    "rii_ratio",
    "inst_ratio",
]

X = df[FEATURES_A]
y = df["success"]


In [23]:
X.shape, y.value_counts(normalize=True)


((561, 6),
 success
 1    0.693405
 0    0.306595
 Name: proportion, dtype: float64)

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape


((448, 6), (113, 6))

In [25]:
from sklearn.ensemble import RandomForestClassifier

rf_A = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=5,
    random_state=42
)

rf_A.fit(X_train, y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [26]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred_A = rf_A.predict(X_test)

print("Option A Accuracy:", accuracy_score(y_test, y_pred_A))
print(confusion_matrix(y_test, y_pred_A))
print(classification_report(y_test, y_pred_A))


Option A Accuracy: 0.7964601769911505
[[17 18]
 [ 5 73]]
              precision    recall  f1-score   support

           0       0.77      0.49      0.60        35
           1       0.80      0.94      0.86        78

    accuracy                           0.80       113
   macro avg       0.79      0.71      0.73       113
weighted avg       0.79      0.80      0.78       113



In [28]:
importance_A = (
    pd.DataFrame({
        "feature": FEATURES_A,
        "importance": rf_A.feature_importances_
    })
    .sort_values(by="importance", ascending=False)
)

importance_A


,feature,importance
5,inst_ratio,0.231482
4,rii_ratio,0.202208
3,hni_ratio,0.175303
2,qib_ratio,0.140324
0,issue_size_crore,0.131580
1,issue_price,0.119102


In [27]:
from pathlib import Path
import joblib

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(rf_A, MODELS_DIR / "rf_option_A.pkl")


['models/rf_option_A.pkl']

In [29]:
importance_A.to_csv(
    "data/processed/option_A_feature_importance.csv",
    index=False
)
